In [1]:
import pandas as pd
df_buy = pd.read_csv('Buy Units Web.csv')
df_rev = pd.read_csv('data_with_sku.csv').drop(['Unnamed: 0'],axis=1).drop_duplicates()
df_rev_org = pd.read_csv('data_with_sku.csv').drop(['Unnamed: 0'],axis=1)
df_rev_org['order_processed_date_pst'] = pd.to_datetime(df_rev_org['order_processed_date_pst'])
df_rel = pd.read_csv('Product Release Dates.csv')
df_cal = pd.read_csv('df_cal.csv')
df_sku_map = pd.read_csv('df_sku_map.csv')
df_sku_map = df_sku_map.drop_duplicates(subset=['sku'], keep='first')
df_sku_map = df_sku_map[df_sku_map['sku']!='FAKE'].reset_index(drop=True)
df_price = pd.read_csv('buy_with_price.csv')
is_seasonal = False

Preparing the revenue data

In [2]:
df_sku_map

,style_id,sku
0,W7055R,W7055R022881
1,MensU3046R,U3046R061295
2,U5006RG,U5012RG100605
3,U5006RG,U5012RG100604
4,M5171R,M5171R060494
...,...,...
56450,N7040F,N7040F000155OZ
56451,A0891U,A0891U08329030
56452,B7US03F,B7US03F11
56453,A0891U,A0891U010105


In [3]:
import numpy as np
"""
This code cleans the family columns in 3 steps:

Removes "drops" values — If any family column contains the word "drops", it's set to NaN.

Removes duplicates across family columns — For each row, if a later family column has the same value as an earlier one (e.g., family_3 == family_1), the later one is set to NaN. The earliest occurrence is kept.

Left-shifts values to fill gaps — After steps 1 and 2 may have created NaN holes in the middle (e.g., family_1=X, family_2=NaN, family_3=Y), this compacts the non-NaN values to the left so there are no gaps (becomes family_1=X, family_2=Y, family_3=NaN).
"""
df_rev['order_processed_date_pst'] = pd.to_datetime(df_rev['order_processed_date_pst'])
df_rev['product_title'] = df_rev['product_title'].str.replace(r'[\'""]', '', regex=True)
df_rev = df_rev.merge(df_sku_map, how='inner', on=['sku'])
families_split = df_rev['families'].str.split(',', expand=True)

# Rename columns to family_1, family_2, etc. (up to 5)
max_cols = min(6, families_split.shape[1])
families_split = families_split.iloc[:, :max_cols]
families_split.columns = [f'family_{i+1}' for i in range(max_cols)]

# Strip whitespace from each column
for col in families_split.columns:
    families_split[col] = families_split[col].str.strip()
    families_split[col] = families_split[col].str.split('-').str[0]
df_rev = pd.concat([df_rev, families_split], axis=1).drop(['families'],axis=1)
df_rev['style_id'] = df_rev['style_id'].str.replace(r'(?i)womens|mens|women|men|womans|mans', '', regex=True)
df_rev = df_rev.apply(lambda col: col.str.lower() if col.dtype == "object" else col)
if is_seasonal:
    df_rev = df_rev[df_rev['planning_status']=='seasonal'].reset_index(drop=True)
family_cols = ['family_1', 'family_2', 'family_3', 'family_4', 'family_5', 'family_6']

for i, col in enumerate(family_cols):
    df_rev[col] = df_rev[col].where(~df_rev[col].str.contains('drops', na=False), other=np.nan)
    for prev_col in family_cols[:i]:
        df_rev[col] = df_rev[col].where(df_rev[col] != df_rev[prev_col], other=np.nan)
arr = df_rev[family_cols].values
mask = pd.isna(arr)
out = np.full_like(arr, np.nan, dtype=object)
for i in range(arr.shape[0]):
    valid = arr[i, ~mask[i]]
    out[i, :len(valid)] = valid
df_rev[family_cols] = out
df_cal['quarter_fy'] = 'q' + df_cal['fiscal_quarter'].astype(str) + '_fy' + (df_cal['fiscal_year'] % 100).astype(str)
df_cal['cal_date'] = pd.to_datetime(df_cal['cal_date'])
df_rev = df_rev.merge(df_cal, how='inner', left_on=['order_processed_date_pst'], right_on=['cal_date'])
df_rev = df_rev.drop(['unique_key','_fivetran_synced','index','rnk'], axis=1)

# --- Apply fallbacks per planning_status bucket independently ---
merge_keys = ['product_title','sku','style_id','planning_status','quarter_fy']

results = []
for ps, df_bucket in df_rev.groupby('planning_status'):
    
    bk = merge_keys
    
    # Family fallback: rows with valid family_1 within this bucket
    fam_lookup = df_bucket[(df_bucket['family_1'].notna()) & (df_bucket['family_1'] != 'no_fam')]\
        [bk + ['family_1']].drop_duplicates(subset=bk, keep='first')\
        .rename(columns={'family_1': 'family_1_alt'})
    
    # Color fallback: rows with valid color within this bucket
    color_lookup = df_bucket[df_bucket['color'].notna()]\
        [bk + ['color']].drop_duplicates(subset=bk, keep='first')\
        .rename(columns={'color': 'color_alt'})
    
    # Color group fallback: rows with valid color_group within this bucket
    cg_lookup = df_bucket[df_bucket['color_group'].notna()]\
        [bk + ['color_group']].drop_duplicates(subset=bk, keep='first')\
        .rename(columns={'color_group': 'color_group_alt'})
    
    # Apply family fallback
    na_mask = df_bucket['family_1'].isna() | (df_bucket['family_1'] == 'no_fam')
    df_na = df_bucket[na_mask].merge(fam_lookup, how='left', on=bk)
    df_ok = df_bucket[~na_mask].copy()
    
    df_b = pd.concat([df_na, df_ok])
    df_b['family_1'] = df_b['family_1'].replace('no_fam', np.nan)
    if 'family_1_alt' in df_b.columns:
        df_b['family_1'] = df_b['family_1'].fillna(df_b['family_1_alt'])
    
    # Apply color fallback
    df_b = df_b.merge(color_lookup, on=bk, how='left')
    df_b['color'] = df_b['color'].fillna(df_b.get('color_alt'))
    
    # Apply color_group fallback
    df_b = df_b.merge(cg_lookup, on=bk, how='left')
    df_b['color_group'] = df_b['color_group'].fillna(df_b.get('color_group_alt'))
    
    results.append(df_b)

df_rev = pd.concat(results, ignore_index=True)

if is_seasonal:
    df_rev = df_rev[df_rev['planning_status']=='seasonal']

# Check results
print(f"family_1 NaN ratio: {(df_rev['family_1'].isna() | (df_rev['family_1']=='no_fam')).sum() / len(df_rev):.4f}")
print(f"color NaN ratio:    {df_rev['color'].isna().sum() / len(df_rev):.4f}")
print(f"color_group NaN ratio: {df_rev['color_group'].isna().sum() / len(df_rev):.4f}")

df_rev = df_rev[['order_processed_date_pst', 'style_id', 'color_group', 'color', 'planning_status',
       'revenue', 'ordered_quantities','family_1', 'family_2', 'family_3', 'family_4', 'quarter_fy','family_5', 'family_6']].drop_duplicates().groupby(['order_processed_date_pst','style_id', 'color_group', 'color', 'planning_status','quarter_fy','family_1', 'family_2', 'family_3', 'family_4', 'family_5', 'family_6'], dropna=False).sum().reset_index()


/var/folders/p2/tb5dw05d6nx3s51cl8jqtfmc0000gp/T/ipykernel_36713/1580862813.py:87: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_b['color_group'] = df_b['color_group'].fillna(df_b.get('color_group_alt'))
/var/folders/p2/tb5dw05d6nx3s51cl8jqtfmc0000gp/T/ipykernel_36713/1580862813.py:77: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_b['family_1'] = df_b['family_1'].replace('no_fam', np.nan)
/var/folders/p2/tb5dw05d6nx3s51cl8jqtfmc0000gp/T/ipykernel_36713/1580862813.py:79: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, 

family_1 NaN ratio: 0.0942
color NaN ratio:    0.0141
color_group NaN ratio: 0.0149


lets work on the buy

In [4]:
df_buy
df_buy = pd.read_csv('Buy Units Web.csv')
df_buy['Style Color'] = df_buy['Style Color'].str.replace(r' X\b', '', regex=True).str.lower()
cols = [' Q1 FY25 ', ' Q2 FY25 ', ' Q3 FY25 ',
       ' Q4 FY25 ', ' Q1 FY26 ']

for col in cols:
    df_buy[col] = pd.to_numeric(df_buy[col].astype(str).str.replace(',', ''), errors='coerce').fillna(0).astype(int)
df_buy['Style Color'] = df_buy['Style Color'].str.replace(r'optic ', '', regex=True)
df_buy['Style Color'] = df_buy['Style Color'].str.replace(r'[\'"”]', '', regex=True)
df_buy.columns = ['style_color', 'style_color_code', 'realigned_code', 'style_code',
       'color_code', 'channel', 'q1_fy25', 'q2_fy25', 'q3_fy25','q4_fy25', 'q1_fy26']
df_price['Time'] = df_price['Time'].str.lower().str.replace(' ', '_')
import pandas as pd


# Pivot price to get one column per season
price_pivot = df_price.pivot_table(
    index='Style Color Code', 
    columns='Time', 
    values='Retail Price', 
    aggfunc='first'
).reset_index()

# Rename by column name, not position
seasons = ['q1_fy25', 'q2_fy25', 'q3_fy25', 'q4_fy25', 'q1_fy26']
price_pivot = price_pivot.rename(columns={s: f'price_{s}' for s in seasons})


# Merge on style_color_code
df_buy = df_buy.merge(price_pivot, left_on='style_color_code', right_on='Style Color Code', how='left').drop(columns='Style Color Code')
df_buy['q1_fy25'] = df_buy['price_q1_fy25']*df_buy['q1_fy25']
df_buy['q2_fy25'] = df_buy['price_q2_fy25']*df_buy['q2_fy25']
df_buy['q3_fy25'] = df_buy['price_q3_fy25']*df_buy['q3_fy25']
df_buy['q4_fy25'] = df_buy['price_q4_fy25']*df_buy['q4_fy25']
df_buy['q1_fy26'] = df_buy['price_q1_fy26']*df_buy['q1_fy26']
df_buy = df_buy.drop(['price_q1_fy25','price_q2_fy25','price_q3_fy25','price_q4_fy25','price_q1_fy26'],axis=1)
df_buy = df_buy[['style_code','q1_fy25','q2_fy25','q3_fy25','q4_fy25','q1_fy26']].groupby(['style_code']).sum().reset_index()
df_buy = df_buy.melt(id_vars='style_code', var_name='quarter_fy', value_name='value')


let's get the release dates

In [5]:

df_rel = df_rel.apply(lambda col: col.str.lower() if col.dtype == "object" else col)

df_rel['Tag'] = df_rel['Tag'].str.split('-').str[0]
df_rel = df_rel[['Tag', 'Early Access','Revenue Investment $ (Total Color Drop)']].dropna()
df_rel.columns = ['Tag','early_access','web_buy']
df_rel['early_access'] = pd.to_datetime(df_rel['early_access'], format='mixed')
df_rel['web_buy'] = (
    df_rel['web_buy']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .apply(lambda x: float(x.replace('m', '')) * 1e6 if 'm' in str(x)
           else float(x.replace('k', '')) * 1e3 if 'k' in str(x)
           else float(x))
)

In [6]:
if is_seasonal:
    df_rev_seasonal = df_rev[df_rev['planning_status']=='seasonal']
else:
    df_rev_seasonal = df_rev.copy()
#df_rev_seasonal = df_rev_seasonal[df_rev_seasonal['color']=='macadamia']
df_rev_seasonal[df_rev_seasonal['family_1'].str.contains('wt25d8',na=False)][['order_processed_date_pst','revenue']].groupby(['order_processed_date_pst']).sum()

,revenue
order_processed_date_pst,
2026-01-08,11309.00
2026-01-09,51370.38
2026-01-10,15280.50
2026-01-11,5871.69
2026-01-12,36468.81
2026-01-13,16999.00
2026-01-14,20860.76
2026-01-15,18413.16
2026-01-16,13305.60


In [19]:
import plotly.express as px
import plotly.graph_objects as go

# ============================================================
filter_for_30d = False  # True = bucket discount-affected families into "other"
launch_window_days = 30  # Number of days to analyze from launch (e.g., 14 or 30)
min_pct_threshold = 1    # Minimum % of daily non-core revenue to show as own line
# ============================================================

exclusion_windows = [
    (pd.to_datetime('2025-10-26'), pd.to_datetime('2025-12-25')),
    (pd.to_datetime('2025-04-26'), pd.to_datetime('2025-05-02')),
]

# --- Aggregate by family_1 ---
df_tmp = df_rev_seasonal.copy()
df_tmp['family_1'] = df_tmp['family_1'].fillna('other')

df_agg = df_tmp.groupby(
    ['order_processed_date_pst', 'family_1', 'planning_status']
)['revenue'].sum().reset_index().rename(columns={'family_1': 'family'})
df_agg['order_processed_date_pst'] = pd.to_datetime(df_agg['order_processed_date_pst'])

# Build release map and web_buy map
df_rel['early_access'] = pd.to_datetime(df_rel['early_access'])
release_map = df_rel.drop_duplicates(subset='Tag').set_index('Tag')['early_access'].to_dict()
web_buy_map = df_rel.drop_duplicates(subset='Tag').set_index('Tag')['web_buy'].to_dict()
df_agg['web_buy'] = df_agg['family'].map(web_buy_map)

if filter_for_30d:
    for win_min, win_max in exclusion_windows:
        df_agg = df_agg[(df_agg['order_processed_date_pst'] < win_min) | (df_agg['order_processed_date_pst'] > win_max)]

    families_to_bucket = set()
    for family, ea_date in release_map.items():
        for win_min, win_max in exclusion_windows:
            if ea_date + pd.Timedelta(days=launch_window_days - 1) >= win_min and ea_date <= win_max:
                families_to_bucket.add(family)

    print(f"Bucketed {len(families_to_bucket)} families into 'other': {sorted(families_to_bucket)}")
    df_agg.loc[df_agg['family'].isin(families_to_bucket), 'family'] = 'other'

    df_other = df_agg[df_agg['family'] == 'other']
    df_rest = df_agg[df_agg['family'] != 'other']
    df_other_agg = df_other.groupby(['order_processed_date_pst', 'planning_status']).agg(
        revenue=('revenue', 'sum'), web_buy=('web_buy', 'sum')
    ).reset_index()
    df_other_agg['family'] = 'other'
    df_agg = pd.concat([df_rest, df_other_agg], ignore_index=True)

    family_min_date = df_agg[df_agg['family'] != 'other'].groupby('family')['order_processed_date_pst'].transform('min')
    df_agg['ea_start'] = df_agg['family'].map(release_map)
    mask = df_agg['ea_start'].isna() & (df_agg['family'] != 'other')
    df_agg.loc[mask, 'ea_start'] = family_min_date[mask]
    df_agg['days_since_launch'] = (df_agg['order_processed_date_pst'] - df_agg['ea_start']).dt.days
    rev_nd = df_agg[df_agg['days_since_launch'].between(0, launch_window_days - 1)]\
        .groupby('family')['revenue'].sum().rename('rev_nd')
    df_agg = df_agg.merge(rev_nd, on='family', how='left')
    df_agg = df_agg.drop(columns=['ea_start', 'days_since_launch'])
    print(f"Remaining families: {df_agg['family'].nunique()} | rev_nd NaN: {df_agg['rev_nd'].isna().sum()} / {len(df_agg)}")

# --- Relabel core rows, then bucket <5% non-core families into 'rest' ---
df_agg.loc[df_agg['planning_status'] == 'core', 'family'] = 'core'

# Compute daily non-core % per family to identify <5% contributors
df_nc = df_agg[df_agg['family'] != 'core'].copy()
df_nc_daily_rev = df_nc.groupby(['order_processed_date_pst', 'family'])['revenue'].sum().reset_index(name='fam_day_rev')
daily_total_nc = df_nc_daily_rev.groupby('order_processed_date_pst')['fam_day_rev'].sum().rename('daily_total')
df_nc_daily_rev = df_nc_daily_rev.merge(daily_total_nc, on='order_processed_date_pst')
df_nc_daily_rev['pct_of_daily'] = (df_nc_daily_rev['fam_day_rev'] / df_nc_daily_rev['daily_total'] * 100)

# Build set of (date, family) pairs below threshold
below_mask = df_nc_daily_rev['pct_of_daily'] < min_pct_threshold
below_keys = set(zip(df_nc_daily_rev.loc[below_mask, 'order_processed_date_pst'],
                      df_nc_daily_rev.loc[below_mask, 'family']))

# Relabel those rows in df_agg as 'rest'
df_agg['_key'] = list(zip(df_agg['order_processed_date_pst'], df_agg['family']))
df_agg.loc[df_agg['_key'].isin(below_keys), 'family'] = 'rest'
df_agg = df_agg.drop(columns=['_key'])

# Re-aggregate rest and core rows
df_agg = df_agg.groupby(
    ['order_processed_date_pst', 'family', 'planning_status']
).agg(revenue=('revenue', 'sum'), web_buy=('web_buy', 'first')).reset_index()

# --- Cumulative revenue as % of web_buy ---
df_agg['web_buy'] = df_agg['family'].map(web_buy_map)
df_agg = df_agg.sort_values(['family', 'order_processed_date_pst'])
_ea_start = df_agg['family'].map(release_map)
_fmin = df_agg.groupby('family')['order_processed_date_pst'].transform('min')
_ea_start = _ea_start.fillna(_fmin)
_from_launch = df_agg['order_processed_date_pst'] >= _ea_start
_rev_for_cum = df_agg['revenue'].where(_from_launch, 0)
df_agg['cum_rev'] = _rev_for_cum.groupby(df_agg['family']).cumsum()
df_agg.loc[~_from_launch, 'cum_rev'] = np.nan
df_agg['pct_web_buy'] = (df_agg['cum_rev'] / df_agg['web_buy'] * 100)

# --- Build df_daily_fam with ranking (derived from df_agg) ---
df_core_daily = df_agg[df_agg['family'] == 'core'].groupby(
    'order_processed_date_pst'
)['revenue'].sum().reset_index()
df_core_daily['family'] = 'core'
df_core_daily['rank'] = -1

df_nc_daily = df_agg[df_agg['family'] != 'core'].groupby(
    ['order_processed_date_pst', 'family']
)['revenue'].sum().reset_index()
daily_total_nc = df_nc_daily.groupby('order_processed_date_pst')['revenue'].sum().rename('daily_total')
df_nc_daily = df_nc_daily.merge(daily_total_nc, on='order_processed_date_pst')
df_nc_daily['pct_of_daily'] = (df_nc_daily['revenue'] / df_nc_daily['daily_total'] * 100).round(1)
df_nc_daily['rank'] = df_nc_daily.sort_values('revenue', ascending=False)\
    .groupby('order_processed_date_pst').cumcount() + 1

df_daily_fam = pd.concat([df_nc_daily, df_core_daily], ignore_index=True)
df_daily_fam = df_daily_fam.sort_values(['order_processed_date_pst', 'rank'])

# --- Plot from df_daily_fam ---
df_daily_fam['web_buy'] = df_daily_fam['family'].map(web_buy_map)
df_daily_fam = df_daily_fam.sort_values(['family', 'order_processed_date_pst'])

_ea_plot = df_daily_fam['family'].map(release_map)
_fmin_plot = df_daily_fam.groupby('family')['order_processed_date_pst'].transform('min')
_ea_plot = _ea_plot.fillna(_fmin_plot)
_from_launch_plot = df_daily_fam['order_processed_date_pst'] >= _ea_plot
_rev_plot = df_daily_fam['revenue'].where(_from_launch_plot, 0)
df_daily_fam['cum_rev'] = _rev_plot.groupby(df_daily_fam['family']).cumsum()
df_daily_fam.loc[~_from_launch_plot, 'cum_rev'] = np.nan
df_daily_fam['pct_web_buy'] = (df_daily_fam['cum_rev'] / df_daily_fam['web_buy'] * 100)

fig = go.Figure()

for family in df_daily_fam['family'].unique():
    df_fam = df_daily_fam[df_daily_fam['family'] == family]
    if family in release_map:
        df_fam = df_fam[df_fam['order_processed_date_pst'] >= release_map[family]]

    is_core = (family == 'core')
    is_rest = (family == 'rest')

    if is_core:
        line_style = dict(color='grey', width=3, dash='dot')
    elif is_rest:
        line_style = dict(color='lightgrey', width=2, dash='dash')
    else:
        line_style = dict()

    fig.add_trace(go.Scatter(
        x=df_fam['order_processed_date_pst'],
        y=df_fam['revenue'],
        mode='lines',
        name=family,
        line=line_style,
        customdata=df_fam[['web_buy','cum_rev','pct_web_buy']].fillna(''),
        hovertemplate=(
            'Date: %{x}<br>Revenue: %{y:,.2f}<br>'
            'Web Buy: %{customdata[0]:,.0f} — Cum Rev: %{customdata[1]:,.0f} (%{customdata[2]:.1f}%)'
            '<extra>%{fullData.name}</extra>'
        )
    ))

fig.update_layout(title='Revenue Over Time by Family (from Early Access)', xaxis_title='Date', yaxis_title='Revenue')
fig.show()


In [20]:
#df_agg.to_csv('df_agg_new.csv')
df_agg = df_agg[df_agg['order_processed_date_pst']!='2026-02-23'] 

In [9]:
df_agg[(~df_agg['family'].isin(['wt25d8','wt25d7','core','sp26d1']))]\
    .groupby(['order_processed_date_pst']).sum().reset_index()[['order_processed_date_pst','revenue']].to_csv('rest.csv')
#['wt25d1', 'wt25d5', 'wt25d7', 'wt25d3', 'wt25d8', 'sp26d1']

In [10]:

df_agg[(df_agg['family'].str.contains('wt25d7')) & (df_agg['order_processed_date_pst']>='2026-01-04') & (df_agg['family']!='core')][['order_processed_date_pst','revenue']].groupby(['order_processed_date_pst']).sum().reset_index().to_csv('wt25d7.csv')
df_agg[(df_agg['family'].str.contains('wt25d8')) & (df_agg['order_processed_date_pst']>='2026-01-18') & (df_agg['family']!='core')][['order_processed_date_pst','revenue']].groupby(['order_processed_date_pst']).sum().reset_index().to_csv('wt25d8.csv')
df_agg[(df_agg['family'].str.contains('sp26d1'))  & (df_agg['order_processed_date_pst']>='2026-02-01') & (df_agg['family']!='core')][['order_processed_date_pst','revenue']].groupby(['order_processed_date_pst']).sum().reset_index().to_csv('sp26d1.csv')
df_agg[(df_agg['family'].str.contains('sp26d2')) & (df_agg['family']!='core')].groupby(['order_processed_date_pst']).sum().reset_index().to_csv('sp26d2.csv')
df_agg[df_agg['family']=='core'].to_csv('core.csv')

df_agg[
    ~(
        ((df_agg['family'].str.contains('wt25d8')) &  (df_agg['family']!='core')) |
        ((df_agg['family'].str.contains('wt25d8')) & (df_agg['family']!='core')) |
        ((df_agg['family'].str.contains('sp26d1')) &  (df_agg['family']!='core')) |
        ((df_agg['family'].str.contains('sp26d2')) & (df_agg['family']!='core')) |
        (df_agg['family']=='core')
    )
][['order_processed_date_pst','revenue']].groupby(['order_processed_date_pst']).sum().reset_index().to_csv('rest.csv')


In [11]:
df_agg[(df_agg['family'].str.contains('sp26d2')) & (df_agg['order_processed_date_pst']>='2026-01-18') & (df_agg['family']!='core')]

,order_processed_date_pst,family,planning_status,revenue,web_buy,cum_rev,pct_web_buy
11529,2026-02-15,sp26d2,seasonal,517916.39,13800000.0,517916.39,3.753017
11555,2026-02-16,sp26d2,seasonal,495855.09,13800000.0,1013771.48,7.346170
11584,2026-02-17,sp26d2,seasonal,367853.88,13800000.0,1381625.36,10.011778
11612,2026-02-18,sp26d2,seasonal,283881.05,13800000.0,1665506.41,12.068887
11637,2026-02-19,sp26d2,seasonal,291316.47,13800000.0,1956822.88,14.179876
11664,2026-02-20,sp26d2,seasonal,253063.90,13800000.0,2209886.78,16.013672
11690,2026-02-21,sp26d2,seasonal,227512.18,13800000.0,2437398.96,17.662311
11717,2026-02-22,sp26d2,seasonal,268512.04,13800000.0,2705911.00,19.608051


In [12]:
dd = df_agg[['order_processed_date_pst','revenue']].groupby(['order_processed_date_pst']).sum().reset_index().tail(29).merge(df_rev_org.drop_duplicates()[['order_processed_date_pst','revenue']].groupby(['order_processed_date_pst']).sum().reset_index().tail(29), how='inner', on = ['order_processed_date_pst'])
#df_agg[(df_agg['family']=='fa25d1') & (df_agg['order_processed_date_pst']>='2025-08-03')][['order_processed_date_pst','revenue']].head(50)
#df_agg.to_csv('df_agg.csv')
dd['err'] = dd['revenue_x']/dd['revenue_y']-1
dd

,order_processed_date_pst,revenue_x,revenue_y,err
0,2026-01-26,4458064.86,4545247.83,-0.019181
1,2026-01-27,4288385.36,4379243.47,-0.020747
2,2026-01-28,4424791.34,4512964.73,-0.019538
3,2026-01-29,4247711.04,4330409.69,-0.019097
4,2026-01-30,4206766.30,4294202.41,-0.020361
5,2026-01-31,4082344.28,4178265.13,-0.022957
6,2026-02-01,5545889.62,5653868.04,-0.019098
7,2026-02-02,4435971.46,4529575.43,-0.020665
8,2026-02-03,4295613.83,4386009.60,-0.020610
9,2026-02-04,4081424.11,4171930.89,-0.021694


In [13]:
df_rev_org = df_rev_org.drop_duplicates()


In [14]:
#d_acc = df_rev_org[(df_rev_org['product_title'].str.lower().str.contains('accolade')) & (df_rev_org['product_title'].str.lower().str.contains('pink')) & (df_rev_org['order_processed_date_pst']>='2026-01-04') & (df_rev_org['order_processed_date_pst']!='2026-02-19')][['order_processed_date_pst','product_title']].drop_duplicates().groupby(['order_processed_date_pst']).count().reset_index()
d_acc = df_rev_org[(~df_rev_org['product_title'].str.lower().str.contains('accolade')) & \
                   #(df_rev_org['product_title'].str.lower().str.contains('hoodie'))  &\
                    (df_rev_org['order_processed_date_pst']>='2026-01-04') & \
                    (df_rev_org['order_processed_date_pst']<='2026-04-04') & \
                    #(df_rev_org['product_title'].str.lower().str.contains('black|white|navy'))  &\
                        (df_rev_org['order_processed_date_pst']!='2026-02-19')].drop_duplicates()[['order_processed_date_pst','revenue']].groupby(['order_processed_date_pst']).sum().reset_index()
#black|white|pink|yellow|grey|espresso|navy

In [23]:
df_agg[df_agg['family']=='sp25d6'].head(40)

,order_processed_date_pst,family,planning_status,revenue,web_buy,cum_rev,pct_web_buy
4053,2025-04-05,sp25d6,seasonal,13649.07,5700000.0,NaN,NaN
4084,2025-04-06,sp25d6,seasonal,951738.75,5700000.0,951738.75,16.697171
4113,2025-04-07,sp25d6,seasonal,848509.00,5700000.0,1800247.75,31.583294
4147,2025-04-08,sp25d6,seasonal,610793.53,5700000.0,2411041.28,42.298970
4182,2025-04-09,sp25d6,seasonal,527052.12,5700000.0,2938093.40,51.545498
4217,2025-04-10,sp25d6,seasonal,496956.02,5700000.0,3435049.42,60.264025
4257,2025-04-11,sp25d6,seasonal,384015.54,5700000.0,3819064.96,67.001140
4299,2025-04-12,sp25d6,seasonal,360440.19,5700000.0,4179505.15,73.324652
4338,2025-04-13,sp25d6,seasonal,499302.57,5700000.0,4678807.72,82.084346
4374,2025-04-14,sp25d6,seasonal,383334.11,5700000.0,5062141.83,88.809506


In [16]:
#d_acc = df_rev_org[(df_rev_org['product_title'].str.lower().str.contains('accolade')) & (df_rev_org['product_title'].str.lower().str.contains('pink')) & (df_rev_org['order_processed_date_pst']>='2026-01-04') & (df_rev_org['order_processed_date_pst']!='2026-02-19')][['order_processed_date_pst','product_title']].drop_duplicates().groupby(['order_processed_date_pst']).count().reset_index()
d_acc = df_rev_org[(df_rev_org['product_title'].str.lower().str.contains('accolade')) & \
                   #(df_rev_org['product_title'].str.lower().str.contains('hoodie'))  &\
                    (df_rev_org['order_processed_date_pst']>='2026-01-04') & \
                    (df_rev_org['order_processed_date_pst']<='2026-04-04') & \
                    #(df_rev_org['product_title'].str.lower().str.contains('black|white|navy'))  &\
                        (df_rev_org['order_processed_date_pst']!='2026-02-19')].drop_duplicates()[['order_processed_date_pst','product_title']].groupby(['order_processed_date_pst']).nunique().reset_index()
#black|white|pink|yellow|grey|espresso|navy

In [68]:
import pandas as pd
df_f = pd.read_csv('~/Downloads/full_cat.csv')
df_sell = pd.read_csv('~/Downloads/sell_through_rate_final.csv')
df_sell['order_date'] = pd.to_datetime(df_sell['order_date'])
df_sell = df_sell.rename(columns={'revenue': 'revenue_sell'})
df_sell['families'] = df_sell['families'].str.lower()
df_f['order_date'] = pd.to_datetime(df_f['order_date'])
df_f['families'] = df_f['families'].str.lower()
df_f = df_f.rename(columns={'revenue': 'revenue_full'})



/var/folders/p2/tb5dw05d6nx3s51cl8jqtfmc0000gp/T/ipykernel_36713/656196123.py:4: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [69]:
df_mm = df_sell.merge(df_f, how='inner', on = ['families', 'order_date'])[['families','planning_status','order_date','revenue_full','revenue_sell']]
df_mm

,families,planning_status,order_date,revenue_full,revenue_sell
0,fa25d2-family,non_core,2025-08-17,5799.19,589722.54
1,fa25d2-family,non_core,2025-08-18,5276.64,498726.40
2,fa25d2-family,non_core,2025-08-19,3596.44,401894.03
3,fa25d2-family,non_core,2025-08-20,4228.77,319899.95
4,fa25d2-family,non_core,2025-08-21,3366.58,331009.48
...,...,...,...,...,...
4486,wt25d8-family,non_core,2026-03-14,3588.89,110339.98
4487,wt25d8-family,non_core,2026-03-15,9172.46,106227.86
4488,wt25d8-family,non_core,2026-03-16,2824.51,103525.81
4489,wt25d8-family,non_core,2026-03-17,8894.00,109108.80


In [41]:
df_sell[df_sell['order_date']=='2026-03-11'][['families','revenue']]['revenue'].sum()

1368466.07

In [43]:
df_f[df_f['order_date']=='2026-03-11'][['revenue','planning_status']].groupby(['planning_status']).sum()

,revenue
planning_status,
Core,857888.01
non_core,2416881.54


In [65]:
df_f

,planning_status,families,order_date,revenue_full
0,non_core,HO24D2-family,2025-01-01,32007.51
1,non_core,SU23MD4-family,2025-01-01,493.10
2,non_core,HO24D1-family,2025-01-02,63740.86
3,non_core,SU24MD2-family,2025-01-03,1393.52
4,non_core,HO22MToffee-family,2025-01-03,5330.69
...,...,...,...,...
193400,non_core,fa21iconic-family,2026-02-28,554.48
193401,non_core,fa21iconic-family,2026-03-01,933.60
193402,non_core,"fa24drops-family,sp25md3-family",2026-03-02,276.00
193403,Core,"ho22d2earlyaccess-family,ho22d2frenchvanilla-f...",2026-03-04,1275.58


,planning_status,families,order_date,revenue
9852,non_core,"fa25drops-family,fa25mbridge-family,sp25drops-...",2026-03-11,1987.80
9853,non_core,"fa25drops-family,fa25mbridge-family",2026-03-11,13754.93
9854,non_core,sp23drops-family,2026-03-11,11616.90
9856,non_core,"su24d4-2-family,su24drops-family,su25d2-family",2026-03-11,257.89
10057,non_core,"fa25drops-family,fa25md3-family",2026-03-11,1742.66
...,...,...,...,...
195854,non_core,fa22d4strawberry-family,2026-03-11,364.45
195953,non_core,"su25drops-family,su25sunsetsneaker-family",2026-03-11,90.00
196069,non_core,"bluemoon-family,marina-family,olivebranch-family",2026-03-11,3141.55
196171,non_core,"bagcollection-family,wt25bagcollection-family",2026-03-11,1200.00
